In [5]:
# ============================================================
# Task 3: Object Detection and Analysis
# COMP6001 Assignment 1
# ============================================================

import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)

import time, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO
from tqdm import tqdm

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ── 1. Paths ─────────────────────────────────────────────────
BLUR_DIR   = Path("/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/blur")
SHARP_DIR  = Path("/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/sharp")
DEBLUR_DIR = Path("/kaggle/working/outputs/task2")
OUTPUT_DIR = Path("/kaggle/working/outputs/task3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "comparisons").mkdir(exist_ok=True)

MAX_SAMPLES = 10
CONF        = 0.25    # lower = catch more detections
IMG_SIZE    = 1280    # match GoPro resolution
IMG_EXTS    = {".png", ".jpg", ".jpeg", ".bmp"}


# ── 2. Load model ─────────────────────────────────────────────
detector = YOLO("yolov8m.pt")   # medium — better accuracy than nano
print("YOLOv8-m loaded.")


# ── 3. Helpers ────────────────────────────────────────────────

def read_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"Cannot read: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def detect(img_rgb):
    """Run YOLO on uint8 RGB image. Returns (detections, latency_s)."""
    t0  = time.perf_counter()
    res = detector(img_rgb, conf=CONF, imgsz=IMG_SIZE, verbose=False)[0]
    lat = time.perf_counter() - t0
    dets = [{"label": detector.names[int(b.cls)],
              "conf":  float(b.conf),
              "xyxy":  b.xyxy[0].tolist()} for b in res.boxes]
    return dets, lat


def draw_boxes(img_rgb, dets, color):
    out = img_rgb.copy()
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["xyxy"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"{d['label']} {d['conf']:.2f}",
                    (x1, max(y1-6, 12)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, color, 1, cv2.LINE_AA)
    return out


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)


def match_count(dets_a, dets_b, iou_thresh=0.5):
    """Count boxes in dets_a that match dets_b (same class + IoU >= thresh)."""
    matched, used = 0, set()
    for da in dets_a:
        for j, db in enumerate(dets_b):
            if j not in used and da["label"] == db["label"] \
               and iou(da["xyxy"], db["xyxy"]) >= iou_thresh:
                matched += 1; used.add(j); break
    return matched


# ── 4. Collect image paths ────────────────────────────────────

blur_paths   = sorted([p for p in BLUR_DIR.rglob("*")
                        if p.suffix.lower() in IMG_EXTS])[:MAX_SAMPLES]
sharp_paths  = sorted([p for p in SHARP_DIR.rglob("*")
                        if p.suffix.lower() in IMG_EXTS])[:MAX_SAMPLES]
deblur_paths = sorted(DEBLUR_DIR.glob("deblurred_*.png"))[:MAX_SAMPLES]

assert len(deblur_paths) == len(blur_paths), (
    f"Found {len(deblur_paths)} deblurred images but {len(blur_paths)} blur images. "
    "Re-run Task 2 first."
)
print(f"Images: {len(blur_paths)} blur | {len(deblur_paths)} deblurred | {len(sharp_paths)} sharp")


# ── 5. Detection loop ─────────────────────────────────────────

results = []

for i, (bp, dp) in enumerate(tqdm(zip(blur_paths, deblur_paths),
                                   total=len(blur_paths), desc="Detecting")):
    blur_rgb   = read_rgb(bp)
    deblur_rgb = read_rgb(dp)
    sharp_rgb  = read_rgb(sharp_paths[i]) if i < len(sharp_paths) else None

    dets_b, lat_b = detect(blur_rgb)
    dets_d, lat_d = detect(deblur_rgb)
    dets_s, lat_s = detect(sharp_rgb) if sharp_rgb is not None else ([], 0)

    results.append({
        "idx":          i,
        "n_blur":       len(dets_b),
        "n_deblur":     len(dets_d),
        "n_sharp":      len(dets_s),
        "conf_blur":    np.mean([d["conf"] for d in dets_b])  if dets_b  else 0.0,
        "conf_deblur":  np.mean([d["conf"] for d in dets_d])  if dets_d  else 0.0,
        "conf_sharp":   np.mean([d["conf"] for d in dets_s])  if dets_s  else 0.0,
        "match_blur":   match_count(dets_b, dets_s),
        "match_deblur": match_count(dets_d, dets_s),
        "lat_blur":     lat_b,
        "lat_deblur":   lat_d,
        "dets_blur":    dets_b,
        "dets_deblur":  dets_d,
        "dets_sharp":   dets_s,
        "_blur":        blur_rgb,
        "_deblur":      deblur_rgb,
        "_sharp":       sharp_rgb,
    })


# ── 6. Side-by-side detection plots ──────────────────────────

BLUE  = (70,  130, 255)
GREEN = (50,  220, 100)
GOLD  = (255, 200,  50)

for r in results:
    ann_b = draw_boxes(r["_blur"],   r["dets_blur"],   BLUE)
    ann_d = draw_boxes(r["_deblur"], r["dets_deblur"], GREEN)
    ann_s = draw_boxes(r["_sharp"],  r["dets_sharp"],  GOLD) \
            if r["_sharp"] is not None else np.zeros_like(ann_b)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor("#111827")
    for ax, img, title in zip(axes,
        [ann_b, ann_d, ann_s],
        [f"Blurred   — {r['n_blur']} dets  conf={r['conf_blur']:.2f}",
         f"Deblurred — {r['n_deblur']} dets  conf={r['conf_deblur']:.2f}",
         f"Sharp GT  — {r['n_sharp']} dets  conf={r['conf_sharp']:.2f}"]
    ):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=10, pad=6)
        ax.axis("off")

    fig.legend(handles=[
        mpatches.Patch(color=np.array(BLUE)/255,  label="Blurred"),
        mpatches.Patch(color=np.array(GREEN)/255, label="Deblurred"),
        mpatches.Patch(color=np.array(GOLD)/255,  label="Sharp GT"),
    ], loc="lower center", ncol=3, fontsize=9,
       facecolor="#1f2937", labelcolor="white", framealpha=0.8)

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(OUTPUT_DIR / "comparisons" / f"detection_{r['idx']:04d}.png",
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

print("Saved detection comparison images.")


# ── 7. Confidence distribution ────────────────────────────────

all_cb = [d["conf"] for r in results for d in r["dets_blur"]]
all_cd = [d["conf"] for r in results for d in r["dets_deblur"]]
all_cs = [d["conf"] for r in results for d in r["dets_sharp"]]

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(CONF, 1.0, 20)
ax.hist(all_cb, bins=bins, alpha=0.6, color="#6c88d4", label=f"Blurred (n={len(all_cb)})")
ax.hist(all_cd, bins=bins, alpha=0.6, color="#e88b3a", label=f"Deblurred (n={len(all_cd)})")
ax.hist(all_cs, bins=bins, alpha=0.6, color="#5dbf85", label=f"Sharp (n={len(all_cs)})")
ax.set_xlabel("Confidence score"); ax.set_ylabel("Count")
ax.set_title("Confidence Score Distribution", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(linewidth=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved confidence distribution.")


# ── 8. Per-class detection counts ────────────────────────────

def class_counts(results, key):
    counts = defaultdict(int)
    for r in results:
        for d in r[key]: counts[d["label"]] += 1
    return counts

cc_b = class_counts(results, "dets_blur")
cc_d = class_counts(results, "dets_deblur")
cc_s = class_counts(results, "dets_sharp")
classes = sorted(set(cc_b) | set(cc_d) | set(cc_s))

x, w = np.arange(len(classes)), 0.28
fig, ax = plt.subplots(figsize=(max(10, len(classes)*1.2), 5))
ax.bar(x-w, [cc_b[c] for c in classes], w, label="Blurred",   color="#6c88d4")
ax.bar(x,   [cc_d[c] for c in classes], w, label="Deblurred", color="#e88b3a")
ax.bar(x+w, [cc_s[c] for c in classes], w, label="Sharp",     color="#5dbf85")
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_ylabel("Total detections")
ax.set_title("Detections per Class", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y", linewidth=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_counts.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved per-class detection plot.")


# ── 9. Detection count & confidence per image ─────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(results))

axes[0].plot(x, [r["n_blur"]   for r in results], color="#6c88d4", marker="o", ms=5, label="Blurred")
axes[0].plot(x, [r["n_deblur"] for r in results], color="#e88b3a", marker="o", ms=5, label="Deblurred")
axes[0].plot(x, [r["n_sharp"]  for r in results], color="#5dbf85", marker="o", ms=5, label="Sharp")
axes[0].set_title("Detections per Image", fontweight="bold")
axes[0].set_xlabel("Image index"); axes[0].set_ylabel("Detection count")
axes[0].legend(); axes[0].grid(linewidth=0.4, alpha=0.5)

axes[1].plot(x, [r["conf_blur"]   for r in results], color="#6c88d4", marker="o", ms=5, label="Blurred")
axes[1].plot(x, [r["conf_deblur"] for r in results], color="#e88b3a", marker="o", ms=5, label="Deblurred")
axes[1].plot(x, [r["conf_sharp"]  for r in results], color="#5dbf85", marker="o", ms=5, label="Sharp")
axes[1].set_title("Mean Confidence per Image", fontweight="bold")
axes[1].set_xlabel("Image index"); axes[1].set_ylabel("Mean confidence")
axes[1].legend(); axes[1].grid(linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "detections_per_image.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved detections per image plot.")


# ── 10. Latency comparison ────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(results))
ax.plot(x, [r["lat_blur"]   for r in results], color="#6c88d4", marker="o", ms=4, label="Blurred")
ax.plot(x, [r["lat_deblur"] for r in results], color="#e88b3a", marker="o", ms=4, label="Deblurred")
ax.set_xlabel("Image index"); ax.set_ylabel("Latency (s)")
ax.set_title("Detection Latency: Blurred vs Deblurred", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(linewidth=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latency.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved latency plot.")


# ── 11. Summary ───────────────────────────────────────────────

n   = len(results)
avg = lambda k: sum(r[k] for r in results) / n

print(f"\n{'='*62}")
print(f"  Task 3 Results — YOLOv8-m  ({n} images)")
print(f"{'='*62}")
print(f"  {'Metric':<30}  {'Blurred':>8}  {'Deblurred':>9}  {'Sharp':>7}")
print(f"  {'-'*58}")
print(f"  {'Avg detections':<30}  {avg('n_blur'):>8.2f}  {avg('n_deblur'):>9.2f}  {avg('n_sharp'):>7.2f}")
print(f"  {'Avg confidence':<30}  {avg('conf_blur'):>8.4f}  {avg('conf_deblur'):>9.4f}  {avg('conf_sharp'):>7.4f}")
print(f"  {'Avg matched vs sharp':<30}  {avg('match_blur'):>8.2f}  {avg('match_deblur'):>9.2f}  {'—':>7}")
print(f"  {'Avg latency (s)':<30}  {avg('lat_blur'):>8.4f}  {avg('lat_deblur'):>9.4f}  {'—':>7}")
print(f"{'='*62}")

print(f"\n  {'idx':>4}  {'#b':>4}  {'#d':>4}  {'#s':>4}  "
      f"{'conf_b':>6}  {'conf_d':>6}  {'match_b':>7}  {'match_d':>7}")
for r in results:
    print(f"  {r['idx']:>4}  {r['n_blur']:>4}  {r['n_deblur']:>4}  {r['n_sharp']:>4}  "
          f"{r['conf_blur']:>6.3f}  {r['conf_deblur']:>6.3f}  "
          f"{r['match_blur']:>7}  {r['match_deblur']:>7}")

Using device: cuda
YOLOv8-m loaded.
Images: 10 blur | 10 deblurred | 10 sharp


Detecting: 100%|██████████| 10/10 [00:02<00:00,  3.76it/s]


Saved detection comparison images.
Saved confidence distribution.
Saved per-class detection plot.
Saved detections per image plot.
Saved latency plot.

  Task 3 Results — YOLOv8-m  (10 images)
  Metric                           Blurred  Deblurred    Sharp
  ----------------------------------------------------------
  Avg detections                      2.50       3.40     5.50
  Avg confidence                    0.3566     0.3689   0.4606
  Avg matched vs sharp                1.70       2.30        —
  Avg latency (s)                   0.0700     0.0576        —

   idx    #b    #d    #s  conf_b  conf_d  match_b  match_d
     0     1     3    15   0.354   0.484        0        3
     1     2     6    17   0.310   0.585        1        6
     2     2     2     5   0.288   0.482        0        0
     3    12    13    11   0.643   0.601       10       10
     4     0     1     1   0.000   0.291        0        0
     5     1     0     1   0.253   0.000        1        0
     6     1     